# Level 4: Domain Adaptation & Specialization

So far, we've focused on making a general-purpose model follow instructions better. But what if you need a model that is an expert in a specific, niche domain like medicine, law, or a particular company's codebase?

This is where **domain adaptation** comes in. The goal is to infuse a general model with deep knowledge of a specific field.

### The Two-Step Process for Deep Domain Expertise

Achieving true domain expertise usually involves two steps:

1.  **Continued Pre-training (Domain Pre-training)**: This is the core of domain adaptation. You take a general pre-trained model (like Llama 3 or Mistral) and continue the pre-training process, but this time, you use a large corpus of text exclusively from your target domain (e.g., all of PubMed for medicine, all of a company's internal documents).

2.  **Task-Specific Fine-Tuning (SFT)**: After the model has learned the language and concepts of the domain, you then perform instruction fine-tuning (like we did with QLoRA in Level 2) on a set of instruction-response pairs specific to tasks *within that domain*. For example, answering medical questions or summarizing legal briefs.

### When is Continued Pre-training Necessary?

Continued pre-training is a significant undertaking. It requires a large, high-quality domain corpus and is computationally expensive (though less so than pre-training from scratch).

You should consider it when:

- The domain has a **highly specialized vocabulary** (e.g., medical terms, legal jargon) that is not well-represented in the general pre-training data.
- The model needs to understand **complex relationships and concepts** unique to the domain.
- Simple SFT is not yielding the required level of performance or accuracy.

For many use cases, starting with a strong base model and performing SFT (and DPO) on a high-quality, in-domain dataset is sufficient.

### Code Example: Setting up Continued Pre-training

Actually running continued pre-training is beyond the scope of a single notebook as it can take days or weeks on multiple GPUs. However, the code below provides a **template** for how you would set it up using the Hugging Face `Trainer` API.

The main difference from SFT is that we use the standard `Trainer` (not `SFTTrainer`) and train on a raw text dataset, where the model's objective is simply to predict the next token in the domain-specific text.

In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import load_dataset

# --- This is a conceptual example. Running this requires a large dataset and significant compute. ---

# 1. Load the base model and tokenizer
model_id = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# 2. Load your domain-specific text corpus
# This should be a large dataset of raw text from your domain (e.g., medical articles, legal documents)
# For this example, we'll use a generic dataset.
domain_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. Tokenize the dataset
def tokenize_function(examples):
    return tokenizer(examples["text"])

tokenized_dataset = domain_dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 4. Set up the data collator
# This will create batches of tokenized text and handle padding.
# The model will be trained to predict the next token.
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. Define Training Arguments
# These would need to be tuned for a real-world scenario.
training_args = TrainingArguments(
    output_dir="./continued-pretraining-results",
    overwrite_output_dir=True,
    num_train_epochs=1, # In reality, this would be much longer
    per_device_train_batch_size=4,
    save_steps=10_000,
    save_total_limit=2,
    prediction_loss_only=True,
    fp16=True, # Assumes a GPU environment
)

# 6. Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset,
)

# 7. Start training (this would take a very long time)
# print("Starting continued pre-training... This is a demo and will not be run to completion.")
# trainer.train() # Uncomment to run, but be aware of resource requirements!

print("Code setup for continued pre-training is complete. In a real scenario, you would now run trainer.train().")